# PancartPlayer — Ablitération gratuite (Colab)

Ce notebook ablitère un modèle GGUF **sans aucun matériel chez vous** :
1. Dépôt du GGUF (téléversement ou Google Drive)
2. GGUF → Safetensors fp16
3. Ablitération (retrait du vecteur de refus)
4. Re-quantification Q4_K_M
5. Téléchargement du résultat

> RAM : un 9B tient sur le runtime **T4 GPU (16 Go)** de Colab gratuit.

In [ ]:
# @title 1. Installation des outils
import os, sys, subprocess, time

!git clone --depth 1 https://github.com/ggerganov/llama.cpp.git
%cd /content/llama.cpp

!pip -q install --upgrade abliterator

print("OK — outils prêts")

## 2. Fournir le GGUF

Deux options :
- Déposer le fichier via l'icône **📁** (onglet Fichiers à gauche) à la racine `/content/`.
- Ou coller une URL Hugging Face : `https://huggingface.co/<auteur>/<repo>/resolve/main/<fichier>.gguf`.

In [ ]:
# @title 2. Choix du fichier
import glob

# Soit le fichier téléversé, soit téléchargement depuis HF :
GGUF_NAME = "ornith-1.5-9b-Q4_K_M.gguf"   # ← à modifier
HF_URL    = ""            # exemple : https://huggingface.co/ornith-ai/.../Ornith-1.5-9B-Q4_K_M.gguf

src = glob.glob("/content/*.gguf")
if src:
    GGUF_PATH = src[0]
elif HF_URL:
    !wget -O /content/"{GGUF_NAME}" "{HF_URL}"
    GGUF_PATH = f"/content/{GGUF_NAME}"
else:
    raise RuntimeError("Téléversez un GGUF ou fournissez HF_URL")

!ls -lh "{GGUF_PATH}"
print("GGUF_PATH =", GGUF_PATH)

In [ ]:
# @title 3. GGUF -> Safetensors fp16
import subprocess

conv = "tools/convert_gguf.py"
if not os.path.exists(conv):
    conv = "tools/gguf/convert_gguf.py"

r = subprocess.run(["python", conv, GGUF_PATH, "--outtype", "f16", "--outfile", "/content/model"],
                   capture_output=True, text=True)
print(r.stdout[-3000:] if r.stdout else r.stderr[-3000:])
print("Conversion OK")

In [ ]:
# @title 4. Ablitération
!python -m abliterator --model /content/model --output /content/model_ablit \
    --prompt "I am sorry, I cannot" \
    --mp-prompt "I am sorry, I cannot"

!ls /content/model_ablit | head
print("Ablitération OK")

In [ ]:
# @title 5. Re-quantification Q4_K_M
!python convert_hf_to_gguf.py /content/model_ablit --outfile /content/ablit.f16.gguf

# Compile de binaire de quantification (rapide) :
!if [ ! -f build/bin/llama-quantize ]; then \
    cmake -B build > /dev/null && cmake --build build -j --target llama-quantize > /dev/null; fi

!build/bin/llama-quantize /content/ablit.f16.gguf /content/abliterated-Q4_K_M.gguf Q4_K_M

!ls -lh /content/abliterated-Q4_K_M.gguf
print("Quantification OK")

In [ ]:
# @title 6. Téléchargement du résultat
from google.colab import files
files.download("/content/abliterated-Q4_K_M.gguf")

# Alternative : monter Google Drive puis copier
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/abliterated-Q4_K_M.gguf /content/drive/MyDrive/